In [ ]:
!pip install -U torch torchvision torchaudio
!pip install -U transformers accelerate peft bitsandbytes datasets scikit-learn pandas json-repair
!pip install trl==0.19.1  # <-- Pin this specific version

In [1]:
import os, json, re, torch, pandas as pd
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoTokenizer,          # replaces AutoProcessor
    AutoModelForCausalLM,   # replaces AutoModelForImageTextToText
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig, DataCollatorForCompletionOnlyLM

## Configurations & Paths

In [2]:
# --- Paths (Matching your ChartCheck Data Structure) ---
TRAIN_PATH = "train_w_spec.json"
VAL_PATH   = "validation_w_spec.json"
TEST1_PATH = "test_1_w_spec.json"
TEST2_PATH = "test_2_w_spec.json"

# --- Model Definitions ---
BASE_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"  # was Qwen2.5-VL-7B-Instruct
OUTPUT_DIR    = "/workspace/qwen15b_verifier_lora"

# Set up caching to persistent volume to prevent RunPod disk crashes
os.environ["HF_HOME"] = "/workspace/huggingface_cache"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## Data Preprocessing & CoT Formulation

In [3]:
from sentence_transformers import SentenceTransformer
semantic_retriever = SentenceTransformer("all-MiniLM-L6-v2")

def retrieve_top_k_series(claim, spec, k=3):
    series_list = spec.get("ser", [])
    if not isinstance(series_list, list) or len(series_list) <= k: return spec
    docs = [claim] + [json.dumps(s) for s in series_list]
    with torch.no_grad():
        embs = semantic_retriever.encode(docs, convert_to_tensor=True)
        sims = torch.nn.functional.cosine_similarity(embs[0:1], embs[1:])
    top_k = sims.topk(k).indices.sort().values.tolist()
    spec["ser"] = [series_list[i] for i in top_k]
    spec["is_truncated"] = True
    return spec

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
def lightweight_prune_for_llm(raw_spec_dict, claim_text):
    """
    Optimized for Generative LLMs with large context windows. 
    Removes structural bloat and irrelevant series, but preserves full data arrays for CoT math.
    """
    if not isinstance(raw_spec_dict, dict): return {}
    cleaned = raw_spec_dict.copy()
    if 'ser' not in cleaned or not isinstance(cleaned['ser'], list) or len(cleaned['ser']) == 0: return {}

    def compress_val(v):
        if isinstance(v, float): return int(v) if v.is_integer() else round(v, 2)
        return v

    def sweep_for_loops(obj):
        """Kills autoregressive VLM hallucinations from Stage 1."""
        if isinstance(obj, dict):
            for k, v in list(obj.items()):
                if k == 'data': continue 
                if isinstance(v, list) and all(isinstance(x, (str, int, float, bool)) for x in v):
                    obj[k] = [compress_val(x) for i, x in enumerate(v) if i == 0 or x != v[i-1]][:20]
                elif isinstance(v, (dict, list)): sweep_for_loops(v)

    # 1. Strip visual/rendering bloat
    cleaned.pop('legend', None)
    cleaned.pop('tooltip', None)
    
    # 2. Sweep to fix JSON hallucination loops
    sweep_for_loops(cleaned)
    
    # 3. Macro-level Semantic Filtering
    cleaned = retrieve_top_k_series(claim_text, cleaned, k=3)
    
    # 4. Clean up surviving series without dropping data points
    # 🛡️ THE FIX: Safely parse topo_type whether it's a string, a list, or None
    topo = cleaned.get('topo', {})
    topo_type_raw = topo.get('type', '') if isinstance(topo, dict) else ''
    
    if isinstance(topo_type_raw, list):
        topo_type = str(topo_type_raw[0]).lower() if len(topo_type_raw) > 0 else ''
    else:
        topo_type = str(topo_type_raw).lower()
        
    for ser in cleaned.get('ser', []):
        if isinstance(ser, dict):
            # Remove metadata that clutters the LLM's prompt
            ser.pop('ds', None); ser.pop('is_subsampled', None); ser.pop('critical_points_retained', None)
            if 'pie' in topo_type: ser.pop('trend', None); ser.pop('stats', None)
            
            # Compress floats to save tokens, but KEEP ALL POINTS for CoT reasoning
            if 'data' in ser and isinstance(ser['data'], list):
                ser['data'] = [[compress_val(v) for v in pt] if isinstance(pt, list) else compress_val(pt) for pt in ser['data']]

    return cleaned

In [5]:
def load_and_format_data(filepath):
    """
    Loads the cached JSONs and formats them into a text-only prompt.
    Forces the model to generate the Rationale BEFORE the Verdict.
    """
    with open(filepath, 'r', encoding='utf-8') as f:
        raw_data = json.load(f)
        
    formatted_data = []
    for item in raw_data:
        claim = item.get("claim", "")
        
        # 1. Extract the raw specification as a DICTIONARY
        spec_dict = item.get("extended_spec", item.get("raw_spec", {}))
        
        # 2. Prune
        cleaned_spec_dict = lightweight_prune_for_llm(spec_dict, claim)
        
        # 3. Convert the PRUNED dictionary into a minified JSON string for the prompt
        cleaned_spec_str = json.dumps(cleaned_spec_dict, separators=(',', ':')) if isinstance(cleaned_spec_dict, dict) else str(cleaned_spec_dict)
                
        # Determine Verdict
        label_val = item.get("label", 0)
        verdict = "SUPPORTED" if label_val == 1 else "REFUTED"

        # Use existing human explanation as the target rationale
        rationale = item.get("explanation", "Based on the chart data, this is the logical conclusion.")
        
        system_prompt = "You are an expert chart fact-checker. Analyze the JSON specification to verify the user's claim."
        user_prompt = f"Verify this claim.\nClaim: {claim}\nChart Specification:\n{cleaned_spec_str}"
        
        # CoT: Rationale first, Verdict last
        assistant_response = f"Rationale: {rationale}\nVerdict: {verdict}"
        
        # Format for Qwen Chat Template
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
            {"role": "assistant", "content": assistant_response}
        ]
        
        formatted_data.append({"messages": messages, "label_val": label_val})
        
    return Dataset.from_pandas(pd.DataFrame(formatted_data))
       

print("Loading and formatting datasets...")
train_dataset = load_and_format_data(TRAIN_PATH)
val_dataset   = load_and_format_data(VAL_PATH)
test1_dataset = load_and_format_data(TEST1_PATH)
test2_dataset = load_and_format_data(TEST2_PATH)
print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test1: {len(test1_dataset)} | Test2: {len(test2_dataset)}")
del semantic_retriever
torch.cuda.empty_cache()

Loading and formatting datasets...
Train: 7607 | Val: 953 | Test1: 939 | Test2: 981


In [7]:
torch.cuda.empty_cache()

## Model Initialization (4-bit Low Resource)

## Training (SFTTrainer)

In [ ]:
import os, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.trainer_utils import get_last_checkpoint
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig, DataCollatorForCompletionOnlyLM

print("Loading Qwen2.5-1.5B-Instruct in bf16 (no quantization needed)...")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # required for SFT loss masking

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.config.use_cache = False
model.gradient_checkpointing_enable()

# LoRA — lower rank is fine for 1.5B, task is narrow
peft_config = LoraConfig(
    r=16,                  # was 64 — 1.5B needs less rank, 7607 samples
    lora_alpha=32,         # keep 2× ratio
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"]
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# Dataset formatting — identical to before, tokenizer.apply_chat_template works the same
def apply_chat_template(example):
    example["text"] = tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )
    return example

train_data = train_dataset.map(apply_chat_template)
val_data   = val_dataset.map(apply_chat_template)

response_template = "<|im_start|>assistant\n"
collator = DataCollatorForCompletionOnlyLM(response_template, tokenizer=tokenizer)

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,

    per_device_train_batch_size=2,   # can increase — 1.5B is much smaller
    gradient_accumulation_steps=8,   # effective batch = 16, same as before
    max_seq_length=4096,             # 1.5B context; spec prose fits well within this
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    per_device_eval_batch_size=4,
    eval_accumulation_steps=4,

    num_train_epochs=4,              # one extra epoch — smaller model benefits
    learning_rate=2e-4,              # higher than 7B — 1.5B can absorb it
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,

    eval_strategy="steps",
    save_strategy="steps",
    eval_steps=150,
    save_steps=150,
    save_total_limit=3,
    logging_steps=10,

    optim="adamw_torch",             # no need for paged_adamw — fits in memory
    fp16=False,
    bf16=True,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    dataset_text_field="text"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    eval_dataset=val_data,
    peft_config=peft_config,
    processing_class=tokenizer,
    data_collator=collator,
    args=training_args,
)

# Verify label masking before training
print("\n--- Verifying Label Masking ---")
sample_text = train_data[0]["text"]
tokenized_sample = tokenizer(sample_text, truncation=True, max_length=3072)
collated = collator([tokenized_sample])
labels = collated["labels"][0].tolist()
tokens = collated["input_ids"][0].tolist()
unmasked = [i for i, l in enumerate(labels) if l != -100]
if unmasked:
    print("Context (ignored):", tokenizer.decode(tokens[:unmasked[0]])[-80:], "...")
    print("Trained on:", tokenizer.decode(tokens[unmasked[0]:unmasked[0]+60]), "...")
else:
    print("WARNING: No unmasked tokens — response_template not matching!")
print("-------------------------------\n")

last_checkpoint = get_last_checkpoint(OUTPUT_DIR) if os.path.isdir(OUTPUT_DIR) else None
print(f"{'Resuming from: ' + last_checkpoint if last_checkpoint else 'Starting fresh'}")

trainer.train(resume_from_checkpoint=last_checkpoint)
model.save_pretrained(f"{OUTPUT_DIR}/best_verifier")
print(f"✅ Done. Adapter saved to {OUTPUT_DIR}/best_verifier")

Loading Qwen2.5-1.5B-Instruct in bf16 (no quantization needed)...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


Map:   0%|          | 0/7607 [00:00<?, ? examples/s]

Map:   0%|          | 0/953 [00:00<?, ? examples/s]

...

Tokenizing eval dataset:   0%|          | 0/953 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/953 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.



--- Verifying Label Masking ---
Context (ignored): e cube inside of which the ball is inscribed."}<|im_end|>
<|im_start|>assistant
 ...
Trained on: Rationale: The chart shows that as the number of dimensions increases, the ratio of the volume of the inscribed ball to the volume of the cube decreases, the curse of dimensionality.
Verdict: SUPPORTED<|im_end|>
 ...
-------------------------------

Resuming from: /workspace/qwen15b_verifier_lora/checkpoint-150


Step,Training Loss,Validation Loss


## Inference & Evaluation Logic

In [ ]:
def evaluate_dataset(dataset, model, processor):
    model.eval()
    true_labels = []
    pred_labels = []
    generated_rationales = []
    
    print(f"Evaluating {len(dataset)} samples...")
    for idx, item in enumerate(dataset):
        # We only want the system and user messages for generation
        prompt_messages = item["messages"][:2] 
        text = processor.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[text], padding=True, return_tensors="pt").to(DEVICE)
        
        with torch.no_grad():
            generated_ids = model.generate(**inputs, max_new_tokens=150, temperature=0.1, do_sample=False)
            
        generated_ids_trimmed = generated_ids[0][len(inputs.input_ids[0]):]
        output_text = processor.decode(generated_ids_trimmed, skip_special_tokens=True).strip()
        
        # Extract Verdict via Regex
        pred_val = 0
        match = re.search(r"Verdict:\s*(SUPPORTED|REFUTED)", output_text, re.IGNORECASE)
        if match:
            if match.group(1).upper() == "SUPPORTED": pred_val = 1
        else:
            # Fallback heuristic if it fails to format
            if "supported" in output_text.lower(): pred_val = 1
            
        true_labels.append(item["label_val"])
        pred_labels.append(pred_val)
        generated_rationales.append(output_text)
        
        if idx % 50 == 0 and idx > 0:
            print(f"Processed {idx}/{len(dataset)}...")

    acc = accuracy_score(true_labels, pred_labels) * 100
    f1 = f1_score(true_labels, pred_labels, average="macro") * 100
    return acc, f1, generated_rationales

# Run evaluation on test splits
print("\nRunning Evaluation on Test1...")
test1_acc, test1_f1, _ = evaluate_dataset(test1_dataset, model, processor)

print("\nRunning Evaluation on Test2...")
test2_acc, test2_f1, _ = evaluate_dataset(test2_dataset, model, processor)

avg_acc = (test1_acc + test2_acc) / 2

## Final ChartCheck Matrix

In [ ]:
# Reconstruct the ChartCheck Matrix formatting
SEP = "=" * 115

print(f"\n{SEP}")
print("  CHARTCHECK EVALUATION MATRIX — Qwen Generative Verifier (Multi-Adapter)")
print(SEP)

matrix_data = {
    "Model": ["DePlot-DeBERTa-class", "ChartSpec + NLI-DeBERTa (old)", "ChartSpec + Qwen-CoT (ours)"],
    "Task": ["C", "C", "C"],
    "Test1_Acc": [75.0, 75.3, round(test1_acc, 1)],
    "Test1_F1":  [75.0, 75.3, round(test1_f1, 1)],
    "Test2_Acc": [72.5, 74.6, round(test2_acc, 1)],
    "Test2_F1":  [72.5, 74.6, round(test2_f1, 1)],
    "Avg_Acc":   [73.8, 75.0, round(avg_acc, 1)]
}

df_final = pd.DataFrame(matrix_data)
print(df_final.to_string(index=False))
print(SEP)

best_baseline_avg = 75.0
delta = avg_acc - best_baseline_avg
direction = "above" if delta >= 0 else "below"
print(f"\n  Avg accuracy vs NLI-DeBERTa baseline : {delta:+.1f}% ({direction})")